# NLP Assignment - 3: Chatbot using Transformers

---

**Objective:** Build a conversational chatbot using a pre-trained transformer model from Hugging Face that can interact with users and generate meaningful responses.

**Model Used:** `microsoft/DialoGPT-medium` - a dialogue-focused model trained specifically for multi-turn conversations.

**Pipeline Flow:**
User Input -> Model Processing -> Response Generation -> Display Output -> Loop Until Exit

---
## Step 1: Install Required Libraries

In [8]:
# Install Hugging Face transformers library and PyTorch backend
!pip install transformers torch --quiet

---
## Step 2: Import Libraries

- `AutoTokenizer` - converts human text into numbers the model can understand
- `AutoModelForCausalLM` - the transformer model that generates text responses
- `torch` - PyTorch, used as the computation engine behind the model

In [4]:
# Importing tokenizer and model classes from Hugging Face transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

# PyTorch is used as the backend for all model computations
import torch

print("Libraries imported successfully!")

Libraries imported successfully!


---
## Step 3: Load the Pre-trained Transformer Model


In [5]:
# We are using DialoGPT-medium - it is specifically fine-tuned for conversations
# This makes it much better at dialogue compared to the base GPT-2 model
MODEL_NAME = "microsoft/DialoGPT-medium"

print(f"Loading model: {MODEL_NAME}")
print("Please wait - model is being downloaded for the first time...")

# Load the tokenizer - handles converting text to token IDs and back
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load the pre-trained model weights from Hugging Face
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Set the model to evaluation mode since we are only generating, not training
model.eval()

print("\nModel loaded successfully! DialoGPT-medium is ready.")

Loading model: microsoft/DialoGPT-medium
Please wait - model is being downloaded for the first time...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


Model loaded successfully! DialoGPT-medium is ready.


---
## Step 4: Define the Response Generation Function

This function handles everything needed to produce a chatbot reply:
1. Encodes the user message into token IDs
2. Appends it to the conversation history so the model remembers context
3. Runs the model to generate a response
4. Decodes the output back into readable text

In [16]:
def generate_response(user_message, conversation_history):

    new_input_ids = tokenizer.encode(
        user_message + tokenizer.eos_token,
        return_tensors="pt"
    )

    if conversation_history is not None:
        bot_input_ids = torch.cat([conversation_history, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.75
    )


    reply = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return reply, chat_history_ids


print("Response generation function defined successfully!")

Response generation function defined successfully!


---
## Step 5: Run the Chatbot

The chatbot will:
- Print a greeting message
- Wait for you to type a message
- Generate and display a response
- Keep looping until you type `exit` or `quit`


In [7]:
def run_chatbot():

    conversation_history = None

    # Print the chatbot header and opening greeting
    print("=" * 55)
    print("   Chatbot using Transformers (DialoGPT-medium)")
    print("=" * 55)
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    print("-" * 55)
    print("  Type 'exit' or 'quit' to end the conversation.")
    print("-" * 55)

    # Keep the chatbot running in a loop until exit condition is met
    while True:

        user_input = input("\nYou: ").strip()

        if not user_input:
            print("Chatbot: Please type something so I can help you!")
            continue

        if user_input.lower() in ["exit", "quit"]:
            print("\nChatbot: It was great talking to you! Goodbye!")
            print("=" * 55)
            break

        response, conversation_history = generate_response(user_input, conversation_history)

        print(f"\nChatbot: {response}")


# Start the chatbot
run_chatbot()

   Chatbot using Transformers (DialoGPT-medium)
Chatbot: Hello! I am your AI assistant. How can I help you today?
-------------------------------------------------------
  Type 'exit' or 'quit' to end the conversation.
-------------------------------------------------------

You: hello


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Chatbot: Hello. Nice to meet you.

You: What is Artificial Intelligence?

Chatbot: A computer program that makes you feel emotion.

You: Who created Python?

Chatbot: It's the programming language of the future.

You: Thank you

Chatbot: It is the programming of the past.

You: exit

Chatbot: It was great talking to you! Goodbye!
